In [1]:
# Load train/val/test splits produced in 01_eda.ipynb (Steps 1-5: drop, impute, leakage audit, split)
import pandas as pd

processed_dir = r"P:\AI\Project_Default\data\processed"

train_df = pd.read_parquet(f"{processed_dir}\\train.parquet")
val_df = pd.read_parquet(f"{processed_dir}\\val.parquet")
test_df = pd.read_parquet(f"{processed_dir}\\test.parquet")

X_train, y_train = train_df.drop(columns=['target']), train_df['target']
X_val, y_val = val_df.drop(columns=['target']), val_df['target']
X_test, y_test = test_df.drop(columns=['target']), test_df['target']

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")


Train: (964040, 71), Val: (206580, 71), Test: (206581, 71)


In [ ]:
# Feature engineering - 5 origination-time features (see notes/02_feat_engineer.md).
# Applied identically to train/val/test via a shared function - no fitting/statistics
# learned from data here, so no train-only-fit concern for these particular features.
import numpy as np

def add_features(df):
    df = df.copy()

    # 1. Loan payment as a share of monthly income
    df['installment_to_income'] = df['installment'] / (df['annual_inc'] / 12)

    # 2. Loan size relative to income
    df['loan_to_income'] = df['loan_amnt'] / df['annual_inc']

    # 3. Credit history length in months (earliest_cr_line to issue_d)
    earliest_cr_line = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y')
    issue_d = pd.to_datetime(df['issue_d'], format='%b-%Y')
    df['credit_history_length'] = (
        (issue_d.dt.year - earliest_cr_line.dt.year) * 12
        + (issue_d.dt.month - earliest_cr_line.dt.month)
    )

    # 4. Revolving credit headroom relative to limit (inf when total_rev_hi_lim == 0).
    # Use np.nan (not pd.NA) so the column stays float64 rather than becoming object dtype.
    df['avail_credit_ratio'] = df['bc_open_to_buy'] / df['total_rev_hi_lim']
    df['avail_credit_ratio'] = df['avail_credit_ratio'].replace([np.inf, -np.inf], np.nan)

    # 5. Combined FICO score
    df['fico_avg'] = (df['fico_range_low'] + df['fico_range_high']) / 2

    return df

X_train = add_features(X_train)
X_val = add_features(X_val)
X_test = add_features(X_test)

new_feature_cols = [
    'installment_to_income', 'loan_to_income', 'credit_history_length',
    'avail_credit_ratio', 'fico_avg',
]
print(X_train[new_feature_cols].dtypes)
print(X_train[new_feature_cols].describe())


In [ ]:
# Clip edge cases in the 3 problematic features - bounds computed on X_train only,
# then applied identically to X_train/X_val/X_test (train-only-fit, avoids leakage
# from val/test statistics; see notes/02_feat_engineer.md edge-case note).
clip_cols = ['installment_to_income', 'loan_to_income', 'credit_history_length']
clip_bounds = {col: X_train[col].quantile(0.99) for col in clip_cols}

for df in (X_train, X_val, X_test):
    for col, upper in clip_bounds.items():
        df[col] = df[col].clip(upper=upper)
    # avail_credit_ratio: NaN from the inf-replacement (total_rev_hi_lim == 0) means
    # "no revolving limit to compare against" - fill with 0 (no available headroom)
    df['avail_credit_ratio'] = df['avail_credit_ratio'].fillna(0)

print("Clip bounds (99th percentile on X_train):")
print(clip_bounds)
print()
print(X_train[new_feature_cols].describe())


In [11]:
# Save feature-engineered splits to data/processed/ for 03_classifier.ipynb
processed_dir = r"P:\AI\Project_Default\data\processed"

X_train.assign(target=y_train).to_parquet(f"{processed_dir}\\train_fe.parquet", index=False)
X_val.assign(target=y_val).to_parquet(f"{processed_dir}\\val_fe.parquet", index=False)
X_test.assign(target=y_test).to_parquet(f"{processed_dir}\\test_fe.parquet", index=False)

print("Saved feature-engineered train/val/test to data/processed/")


Saved feature-engineered train/val/test to data/processed/
